# ClickHouse

Clickhouse is an open-source OLAP DBMS. This DBMS specialises in processing large ammounts of data and is primarily used by analytics and ML teams to extract data from huge datasets.

## Data organisation

This section describes the ideas, features and syntax of data organisation in ClickHouse

There are following essesnces important to know:

- **Tables**.
- **Databases**.
- **Views**.

**Note** in clickhouse there is **no schemas**, in post external tools the databases are considered as schemas.

Check more in the [Data organisation](clickhouse/data_organisation.ipynb).

## Parts

The clickhouse stores data as parts. It manipulates these parts over time. Information about the current parts is stored in the `system.parts` table.

The practically important columns for the `system.parts` table are:

- `name`: the name of the part, there is encoded some iformation about the part.
- `table`: describes to which table part is belongs to.
- `active`: colum takes value 1 if the corresponding part actually is used.

---

The following cell displays the description for the `system.parts` table.

In [2]:
--ClickHouse
DESCRIBE TABLE system.parts;

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
partition,String,,,The partition name.,,
name,String,,,"Name of the data part. The part naming structure can be used to determine many aspects of the data, ingest, and merge patterns. The part naming format is the following: ``` <partition_id>_<minimum_block_number>_<maximum_block_number>_<level>_<data_version> ``` * Definitions: - `partition_id` - identifies the partition key - `minimum_block_number` - identifies the minimum block number in the part. ClickHouse always merges continuous blocks - `maximum_block_number` - identifies the maximum block number in the part - `level` - incremented by one with each additional merge on the part. A level of 0 indicates this is a new part that has not been merged. It is important to remember that all parts in ClickHouse are always immutable - `data_version` - optional value, incremented when a part is mutated (again, mutated data is always only written to a new part, since parts are immutable)",,
uuid,UUID,,,The UUID of data part.,,
part_type,String,,,The data part storing format. Possible Values: Wide (a file per column) and Compact (a single file for all columns).,,
active,UInt8,,,"Flag that indicates whether the data part is active. If a data part is active, it's used in a table. Otherwise, it's about to be deleted. Inactive data parts appear after merging and mutating operations.",,
marks,UInt64,,,"The number of marks. To get the approximate number of rows in a data part, multiply marks by the index granularity (usually 8192) (this hint does not work for adaptive granularity).",,
rows,UInt64,,,The number of rows.,,
files,UInt64,,,The number of files in the data part.,,
bytes_on_disk,UInt64,,,Total size of all the data part files in bytes.,,
data_compressed_bytes,UInt64,,,"Total size of compressed data in the data part. All the auxiliary files (for example, files with marks) are not included.",,


## MergeTree

The **MergeTree** is the most popular engine in clickhouse that determines how ClickHouse operates with data.

There are different variations of the MergeTree engines that implement different features, they all follow same data organization idea.

- **[MergeTree](https://clickhouse.com/docs/engines/table-engines/mergetree-family/mergetree)**: classical option without additional features. 
- **[ReplacingMergeTree](https://clickhouse.com/docs/engines/table-engines/mergetree-family/replacingmergetree)**: replaces the rows with same values for order by key.

For more check:

- [ClickHouse: deepdive into MergeTree](https://habr.com/ru/articles/1001054/)(rus) page.
- The [MergeTree engine family](https://clickhouse.com/docs/engines/table-engines/mergetree-family) page.

### Replacing

The ReplacingMergeTree engine specifies that the table should have only one combination of the order-by key. Deduplication happens during parts merging in the ClickHouse background. The merging process cannot be controlled; apply the `OPTIMIZE` to the table to force ClickHouse to perform deduplication.

As the argument of the engine constructor, specify the name tha would be interpreted as version column. If there are two records with the same order-by values, the merging algorithm will keep the one with the highest value in the `version` column.

Check more in the:

- [ReplacingMergeTree table engine](https://clickhouse.com/docs/engines/table-engines/mergetree-family/replacingmergetree) reference.
- [Working with the ReplacingMergeTree engine](https://clickhouse.com/docs/guides/replacing-merge-tree) tutorial.

---

The following cell defines the table that uses `ReplacingMergeTree` engine:

- The `id` column is the order-by key, specifies unique elements.
- The `version` column is the column that specifies the version of the record.

In [33]:
--ClickHouse
DROP TABLE IF EXISTS temp_table;
CREATE TABLE temp_table (
    id UInt8,
    version UInt8
)
ENGINE = ReplacingMergeTree(version)
ORDER BY id;

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,8390857,1118511,d1089d9b-4c08-4b34-a452-8258a2bc9e6c


The following cell insert some values into the table:

In [34]:
--ClickHouse
INSERT INTO temp_table (*) VALUES
    (1, 1),
    (2, 2);

The other records with same ids but different verions:

In [35]:
--ClickHouse
INSERT INTO temp_table (*) VALUES
    (1, 2),
    (2, 1);
SELECT * FROM temp_table;

id,version
1,2
2,1
1,1
2,2


Just after insert all rows are present in the table. The following cell show the table after `OPTIMIZE` applied to it:

In [36]:
--ClickHouse
OPTIMIZE TABLE temp_table;
SELECT * FROM temp_table;

id,version
1,2
2,2


The output now only includes rows with unique id and greates version.

## Primary key

Primary key in clickhouse have differnt properties from other DBMS's.

In clichouse, the primary key does not required to have unique values, mainly used as "primary inde". The table is sorted and separted into granules accordingly to the primary keys. This is due to the architecture of clickhouse.

It is a good idea to choose the columns with low cardinality (i.e. small number of unique values) as the primary key. ClickHouse uses these columns to separte data internally data into granules, which are then used quckly query the data.

---

The following cell creates the table, and specifies `id` to be the primary key:

In [16]:
--ClickHouse
DROP TABLE IF EXISTS pk_example;
CREATE TABLE pk_example(
    id INT,
    val TEXT
)
ENGINE = MergeTree()
PRIMARY KEY (id);

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,37294218,1119215,490ec853-80ab-45c1-a0bf-267ab55f8730


Inserting some data to clickhouse:

In [17]:
--ClickHouse
INSERT INTO pk_example VALUES (2, 'some extra'), (1, 'data1'), (1, 'data2');

Note that it is ok to enter the same values for the column that is is defined as the primary key, and the data will be sorted accodingly:

In [18]:
--ClickHouse
SELECT * FROM pk_example;

id,val
1,data1
1,data2
2,some extra


### Optimization

In clichouse, the primary key sorts the data and separates it to granules. If you specify a primary key in a query filter, the engine will only check those granules that begong to that primary key value.

---

The following cell creates a data table containing fields `id` and `category`, with `category` is defined as the primary key.

In [111]:
--ClickHouse
DROP TABLE IF EXISTS optim_example;
CREATE TABLE optim_example(
    id INT,
    category String
)
PRIMARY KEY category;

INSERT INTO optim_example
SELECT
    number AS id,
    CASE
        WHEN number < 25000 THEN 'cat'
        WHEN number < 50000 THEN 'dog'
        WHEN number < 75000 THEN 'mouse'
        ELSE 'elephant'
    END
FROM numbers(100000);

read_rows,read_bytes,written_rows,written_bytes,total_rows_to_read,result_rows,result_bytes,elapsed_ns,memory_usage,query_id
0,0,0,0,0,0,0,28296947,1119159,147e2090-f262-4ca4-acc7-7a9fd99b594a


**Note**: It may be confusing that the primary key is not the `id` column, but the column that contains the some kind of category. Keep in mind that the primary key in clickhouse have different meaning.

The following table shows the distribution of the categories:

In [110]:
--ClickHouse
SELECT
    category,
    COUNT(*)
FROM optim_example
GROUP BY category;

category,COUNT()
elephant,250000
dog,250000
cat,250000
mouse,250000


Now consider how many rows are read to execute query:

```sql
select * from optim_example where id=12345;
```

```
   ┌────id─┬─category─┐
1. │ 12345 │ cat      │
   └───────┴──────────┘

1 row in set. Elapsed: 0.003 sec. Processed 100.00 thousand rows, 449.15 KB (33.43 million rows/s., 150.16 MB/s.)
Peak memory usage: 165.52 KiB.
```

**Note** `Processed 100.00 thousand rows`, which means they query needs to check every row in the table.

And query to the same table which filters by the primary key:

```sql
select * from optim_example where category='mouse';
```

```
...
25000 rows in set. Elapsed: 0.003 sec. Processed 26.27 thousand rows, 319.08 KB (8.22 million rows/s., 99.83 MB/s.)
Peak memory usage: 581.20 KiB.
```

A much lower number of rows were read by the computer - clickhouse worked only with the granules corresponding to the `category='mouse'`.

## Data types

Check the available ClickHouse datatypes the system table `system.data_type_family`.

Some types are just alices to the other ones.

For more:

- Check the description of the different datatypes in clickhouse in the corresponding [reference page](https://clickhouse.com/docs/sql-reference/data-types).
- The [Data types](clickhouse/data_types.ipynb) page.

---

The following cell shows the contents of the `system.data_type_family` table.

In [3]:
--ClickHouse
SELECT * FROM system.data_type_families LIMIT 10;

name,case_insensitive,alias_to
JSON,1,
Dynamic,0,
Geometry,0,
Polygon,0,
Ring,0,
Point,0,
SimpleAggregateFunction,0,
IntervalYear,0,
IntervalMonth,0,
IntervalWeek,0,


## Functions

There is a set of prebuilt functions in ClickHouse.

Generally clickhouse have the same group of functions as any other dialect:

- Regular Functions: Whose result for each row is independent from other rows.
- Aggregate Functions: Functions that accumulate a set of values across rows.
- Table Functions: Methods for constructing tables.
- Window Functions: Functions which let you perform calculations across a set of rows that are related to the current row.


Check more in the: 

- The [Functions Overview](https://clickhouse.com/docs/sql-reference/functions) page.
- The [Functions](clickhouse/functions.ipynb) page.

## Statements

ClickHouse is really specific SQL dialect. This section considers the features of the statements in the ClickHouse.

For more check the:

- [ClickHouse SQL Statements](https://clickhouse.com/docs/sql-reference/statements) reference page.
- [Statements](clickhouse/statements.ipynb) page.